# 1. Libraries Import

This notebook standardizes all locally available datasets into a shared text-classification schema with labels `['A1-A2', 'B1-B2', 'C1-C2']` and saves parquet outputs to `data/prepared/`.


In [1]:
from __future__ import annotations

import importlib.util
import re
import unicodedata
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

UNIFIED_LABELS = ["A1-A2", "B1-B2", "C1-C2"]
DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def get_parquet_engine() -> str:
    for engine in ("pyarrow", "fastparquet"):
        if importlib.util.find_spec(engine) is not None:
            return engine
    raise ImportError(
        "Writing parquet files requires `pyarrow` or `fastparquet` in the notebook kernel."
    )

PARQUET_ENGINE = get_parquet_engine()
print(f"Using parquet engine: {PARQUET_ENGINE}")

CYRILLIC_LOOKALIKES = str.maketrans({"с": "c", "С": "C"})

def normalize_slug(value: str) -> str:
    normalized = value.translate(CYRILLIC_LOOKALIKES)
    normalized = unicodedata.normalize("NFKD", normalized)
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    return normalized.replace("_", "-").lower()

def resolve_dataset_dir(slug: str) -> Path:
    exact_path = DATA_DIR / slug
    if exact_path.exists():
        return exact_path

    normalized_target = normalize_slug(slug)
    candidates = [
        path for path in DATA_DIR.iterdir()
        if path.is_dir() and normalize_slug(path.name) == normalized_target
    ]
    if candidates:
        return candidates[0]

    fuzzy_candidates = [
        path for path in DATA_DIR.iterdir()
        if path.is_dir() and normalized_target in normalize_slug(path.name)
    ]
    if fuzzy_candidates:
        return fuzzy_candidates[0]

    raise FileNotFoundError(f"Could not resolve dataset directory for: {slug}")

def natural_sort_key(path: Path) -> list[object]:
    parts = re.split(r"(\d+)", path.stem)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def read_text_file(path: Path) -> str:
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return path.read_text(encoding=encoding).strip()
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("utf-8", b"", 0, 1, f"Unable to decode {path}")

def infer_observation_format(series: pd.Series, word_threshold: int = 3, sample_size: int = 200) -> str:
    sample = series.fillna("").astype(str).str.strip()
    sample = sample[sample.ne("")].head(sample_size)
    if sample.empty:
        return "unknown"
    token_counts = sample.str.split().str.len()
    if (token_counts <= word_threshold).mean() >= 0.85:
        return "isolated_words"
    return "full_texts_or_excerpts"

def summarize_text_lengths(series: pd.Series) -> pd.Series:
    cleaned = series.fillna("").astype(str).str.strip()
    cleaned = cleaned[cleaned.ne("")]
    token_counts = cleaned.str.split().str.len()
    return token_counts.describe(percentiles=[0.25, 0.50, 0.75]).round(2)

def show_basic_overview(df: pd.DataFrame, text_col: str, label_cols: list[str]) -> None:
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Text column:", text_col)
    print("Label columns:", label_cols)
    print("Observation format:", infer_observation_format(df[text_col]))
    print("Text length statistics (word counts):")
    display(summarize_text_lengths(df[text_col]))

    for label_col in label_cols:
        if label_col in df.columns:
            print(f"Value counts for {label_col}:")
            display(df[label_col].value_counts(dropna=False).rename("count"))

    preview_cols = [text_col] + [col for col in label_cols if col in df.columns]
    preview = df[preview_cols].head(3).copy()
    preview[text_col] = (
        preview[text_col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.slice(0, 200)
    )
    print("Preview:")
    display(preview)

def build_standardized_dataframe(
    df: pd.DataFrame,
    text_col: str,
    label_col: str,
    source_dataset: str,
) -> pd.DataFrame:
    standardized = df[[text_col, label_col]].copy()
    standardized = standardized.rename(columns={text_col: "text", label_col: "label"})
    standardized["source_dataset"] = source_dataset
    return standardized

def save_standardized_dataset(df: pd.DataFrame, output_name: str) -> tuple[pd.DataFrame, Path]:
    standardized = df.loc[:, ["text", "label", "source_dataset"]].copy()
    standardized["text"] = (
        standardized["text"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    standardized["label"] = standardized["label"].astype("string").str.strip()
    standardized = standardized[
        standardized["text"].ne("") & standardized["label"].isin(UNIFIED_LABELS)
    ].reset_index(drop=True)

    output_path = OUTPUT_DIR / output_name
    standardized.to_parquet(output_path, index=False, engine=PARQUET_ENGINE)

    print(f"Saved {output_path} with shape {standardized.shape}")
    print("Unified label distribution:")
    display(standardized["label"].value_counts().reindex(UNIFIED_LABELS, fill_value=0))
    display(standardized.head(3))
    return standardized, output_path

SCORE_CLASS_BANDS = {
    "A1-A2": (0.0, 1 / 3),
    "B1-B2": (1 / 3, 2 / 3),
    "C1-C2": (2 / 3, 1.0),
}

def min_max_normalize(series: pd.Series) -> pd.Series:
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(pd.NA, index=series.index, dtype="Float64")
    if min_value == max_value:
        return pd.Series([0.5] * len(series), index=series.index, dtype="Float64")
    return (series - min_value) / (max_value - min_value)

def normalized_score_to_label(value: float | pd._libs.missing.NAType) -> str | pd._libs.missing.NAType:
    if pd.isna(value):
        return pd.NA
    for label, (lower_bound, upper_bound) in SCORE_CLASS_BANDS.items():
        if value <= upper_bound:
            return label
    return pd.NA

# This free, local generator is a fallback for word-level datasets.
# The rule-based version is effectively instant for thousands of rows.
# Replacing it with a local LLM or remote API can increase runtime from seconds to minutes or hours.
def generate_rule_based_sentence(word: str, unified_label: str) -> str:
    word = str(word).strip()
    templates = {
        "A1-A2": f"This short example uses the word '{word}' in a simple sentence.",
        "B1-B2": f"In an everyday context, the word '{word}' appears naturally in this sentence.",
        "C1-C2": f"In a more formal and abstract context, the word '{word}' is used precisely in this sentence.",
    }
    return templates.get(unified_label, f"The word '{word}' appears in this example sentence.")

def build_synthetic_sentence_dataset(
    words_df: pd.DataFrame,
    word_col: str,
    label_col: str,
    source_dataset: str,
    output_name: str,
    generator=generate_rule_based_sentence,
) -> tuple[pd.DataFrame, Path]:
    synthetic_df = words_df[[word_col, label_col]].copy()
    synthetic_df["text"] = synthetic_df.apply(
        lambda row: generator(row[word_col], row[label_col]),
        axis=1,
    )
    synthetic_df["source_dataset"] = source_dataset
    return save_standardized_dataset(
        synthetic_df[["text", label_col, "source_dataset"]].rename(columns={label_col: "label"}),
        output_name,
    )

LEXILE_CLASS_BANDS = {
    "A1-A2": {"max": 700},
    "B1-B2": {"min": 700, "max": 1100},
    "C1-C2": {"min": 1100},
}

def extract_lexile_value(value: object) -> float | pd._libs.missing.NAType:
    if pd.isna(value):
        return pd.NA
    numbers = [int(match) for match in re.findall(r"\d+", str(value))]
    if not numbers:
        return pd.NA
    return float(sum(numbers) / len(numbers))

def lexile_to_label(value: float | pd._libs.missing.NAType) -> str | pd._libs.missing.NAType:
    if pd.isna(value):
        return pd.NA
    if value <= LEXILE_CLASS_BANDS["A1-A2"]["max"]:
        return "A1-A2"
    if value <= LEXILE_CLASS_BANDS["B1-B2"]["max"]:
        return "B1-B2"
    return "C1-C2"

def load_folder_corpus(root_dir: Path, folder_names: list[str], label_column_name: str) -> pd.DataFrame:
    rows = []
    for folder_name in folder_names:
        folder_path = root_dir / folder_name
        if not folder_path.exists():
            raise FileNotFoundError(f"Expected folder not found: {folder_path}")
        for text_path in sorted(folder_path.glob("*.txt"), key=natural_sort_key):
            rows.append(
                {
                    "document_id": f"{folder_name}/{text_path.stem}",
                    label_column_name: folder_name,
                    "text": read_text_file(text_path),
                }
            )
    return pd.DataFrame(rows)

def load_simple_xlsx_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
    namespace = {
        "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
        "doc": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    }

    def column_index(column_letters: str) -> int:
        index = 0
        for character in column_letters:
            if character.isalpha():
                index = index * 26 + (ord(character.upper()) - 64)
        return index - 1

    with ZipFile(path) as archive:
        shared_strings = []
        if "xl/sharedStrings.xml" in archive.namelist():
            shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            for item in shared_root:
                text_parts = [
                    text_node.text or ""
                    for text_node in item.iter("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")
                ]
                shared_strings.append("".join(text_parts))

        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        sheets = workbook.find("main:sheets", namespace)
        relationships = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        relationship_map = {
            relationship.attrib["Id"]: relationship.attrib["Target"]
            for relationship in relationships.findall("rel:Relationship", namespace)
        }

        target_sheet = None
        for sheet in sheets:
            if sheet.attrib.get("name") == sheet_name:
                relationship_id = sheet.attrib[
                    "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
                ]
                target_sheet = "xl/" + relationship_map[relationship_id]
                break
        if target_sheet is None:
            raise KeyError(f"Sheet '{sheet_name}' not found in {path}")

        sheet_root = ET.fromstring(archive.read(target_sheet))
        rows = []
        for row in sheet_root.findall(".//main:sheetData/main:row", namespace):
            values_by_position = {}
            for cell in row.findall("main:c", namespace):
                reference = cell.attrib.get("r", "")
                column_letters = "".join(character for character in reference if character.isalpha())
                position = column_index(column_letters)
                cell_type = cell.attrib.get("t")
                value = None

                raw_value = cell.find("main:v", namespace)
                if raw_value is not None:
                    value = shared_strings[int(raw_value.text)] if cell_type == "s" else raw_value.text

                inline_string = cell.find("main:is", namespace)
                if inline_string is not None:
                    text_parts = [
                        text_node.text or ""
                        for text_node in inline_string.iter(
                            "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t"
                        )
                    ]
                    value = "".join(text_parts)

                values_by_position[position] = value

            if values_by_position:
                width = max(values_by_position) + 1
                rows.append([values_by_position.get(index) for index in range(width)])

    if not rows:
        return pd.DataFrame()

    width = max(len(row) for row in rows)
    normalized_rows = [row + [None] * (width - len(row)) for row in rows]
    header = normalized_rows[0]
    data = normalized_rows[1:]
    return pd.DataFrame(data, columns=header)

def load_clear_corpus(path: Path, sheet_name: str = "Data") -> pd.DataFrame:
    if importlib.util.find_spec("openpyxl") is not None:
        return pd.read_excel(path, sheet_name=sheet_name)
    print("openpyxl is not available; using the XML fallback loader for CLEAR_corpus_final.xlsx.")
    return load_simple_xlsx_sheet(path, sheet_name=sheet_name)

DATASET_DIRS = {
    name: resolve_dataset_dir(name)
    for name in [
        "uk-key-stage-readability-for-english-texts",
        "asap-2-0",
        "asap-aes",
        "cerf-levelled-english-texts",
        "one-stop-corpus-english",
        "cambridge-english-readability",
        "commonlit-readability-prize",
    ]
}
print("Resolved dataset directories:")
for name, path in DATASET_DIRS.items():
    print(f"- {name}: {path}")

Using parquet engine: pyarrow
Resolved dataset directories:
- uk-key-stage-readability-for-english-texts: data/uk-key-stage-readability-for-english-texts
- asap-2-0: data/asap-2-0
- asap-aes: data/asap-aes
- cerf-levelled-english-texts: data/cerf-levelled-english-texts
- one-stop-corpus-english: data/one-stop-corpus-english
- cambridge-english-readability: data/cambridge-english-readability
- commonlit-readability-prize: data/сommonlit-readability-prize


# 2. uk-key-stage-readability-for-english-texts

This dataset contains English reading passages labeled by UK Key Stage, making it useful as a coarse readability corpus.

- Structure: two balanced CSV files (`TRAIN_balanced.csv` and `TEST_balanced.csv`) with one row per passage; the text is stored in `segments`.
- Observations: 20,000 rows are available locally (16,000 train + 4,000 test).
- Labels: `UK Key Stage` with four native classes: `Key Stage 2 (KS2)`, `Key Stage 3 (KS3)`, `Key Stage 4 (KS4)`, and `Key Stage 5 (KS5)`.
- Format: full text passages/excerpts, not isolated words.


In [2]:
uk_dir = DATASET_DIRS["uk-key-stage-readability-for-english-texts"]

uk_train = pd.read_csv(uk_dir / "TRAIN_balanced.csv")
uk_test = pd.read_csv(uk_dir / "TEST_balanced.csv")
uk_raw = pd.concat(
    [uk_train.assign(split="train"), uk_test.assign(split="test")],
    ignore_index=True,
)

text_col = "segments"
original_label_col = "UK Key Stage"

show_basic_overview(uk_raw, text_col=text_col, label_cols=[original_label_col, "split"])
display(uk_raw[["char_count", "word_count", "sentence_count"]].describe().round(2))

uk_label_mapping = {
    "Key Stage 2 (KS2)": "A1-A2",
    "Key Stage 3 (KS3)": "B1-B2",
    "Key Stage 4 (KS4)": "B1-B2",  # Assumption: KS3 and KS4 both fit the middle difficulty band.
    "Key Stage 5 (KS5)": "C1-C2",
}
print("Unified mapping:", uk_label_mapping)

uk_prepared = uk_raw.copy()
uk_prepared["label"] = uk_prepared[original_label_col].map(uk_label_mapping)
assert uk_prepared["label"].notna().all(), "Unmapped UK Key Stage labels found."

uk_prepared = build_standardized_dataframe(
    uk_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="uk-key-stage-readability-for-english-texts",
)
uk_standardized, uk_output_path = save_standardized_dataset(
    uk_prepared,
    "uk-key-stage-readability-for-english-texts.parquet",
)


Shape: (20000, 73)
Columns: ['title', 'author', 'category', 'UK Key Stage', 'segments', 'char_count', 'word_count', 'sentence_count', 'avg_word_length', 'avg_sentence_length', 'type_token_ratio', 'pronoun_freq', 'function_words_count', 'punctuation_frequency', 'sentiment_polarity', 'sentiment_subjectivity', 'readability grades_Kincaid', 'readability grades_ARI', 'readability grades_Coleman-Liau', 'readability grades_FleschReadingEase', 'readability grades_GunningFogIndex', 'readability grades_LIX', 'readability grades_SMOGIndex', 'readability grades_RIX', 'readability grades_DaleChallIndex', 'sentence info_characters_per_word', 'sentence info_syll_per_word', 'sentence info_words_per_sentence', 'sentence info_sentences_per_paragraph', 'sentence info_type_token_ratio', 'sentence info_characters', 'sentence info_syllables', 'sentence info_words', 'sentence info_wordtypes', 'sentence info_sentences', 'sentence info_paragraphs', 'sentence info_long_words', 'sentence info_complex_words', 'se

count    20000.00
mean        78.48
std         17.77
min          4.00
25%         71.00
50%         81.00
75%         89.00
max        716.00
Name: segments, dtype: float64

Value counts for UK Key Stage:


UK Key Stage
Key Stage 5 (KS5)    5000
Key Stage 2 (KS2)    5000
Key Stage 3 (KS3)    5000
Key Stage 4 (KS4)    5000
Name: count, dtype: int64

Value counts for split:


...

,segments,UK Key Stage,split
0,"Wherefore I liken love nowadays unto summer and winter; for like as the one is hot and the other cold, so fareth love nowadays; therefore all ye that be lov...",Key Stage 5 (KS5),train
1,"I seriously advise you to write, by my hands, offering the hospitality of your house (and heart), and the hospitality of my house (and heart), to that injur...",Key Stage 2 (KS2),train
2,"Wolves or watch-dogs, it was hard to say from which the sheep had most to fear. The Castle of Villefranche was harsh and stern as its master. A broad moat, ...",Key Stage 5 (KS5),train


,char_count,word_count,sentence_count
count,20000.00,20000.00,20000.00
mean,440.40,95.00,4.74
std,101.12,23.34,3.27
min,34.00,6.00,1.00
25%,402.00,83.00,2.00
50%,460.00,97.00,4.00
75%,494.00,109.00,6.00
max,5258.00,941.00,53.00


Unified mapping: {'Key Stage 2 (KS2)': 'A1-A2', 'Key Stage 3 (KS3)': 'B1-B2', 'Key Stage 4 (KS4)': 'B1-B2', 'Key Stage 5 (KS5)': 'C1-C2'}
Saved data/prepared/uk-key-stage-readability-for-english-texts.parquet with shape (20000, 3)
Unified label distribution:


label
A1-A2     5000
B1-B2    10000
C1-C2     5000
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"Wherefore I liken love nowadays unto summer and winter; for like as the one is hot and the other cold, so fareth love nowadays; therefore all ye that be lov...",C1-C2,uk-key-stage-readability-for-english-texts
1,"I seriously advise you to write, by my hands, offering the hospitality of your house (and heart), and the hospitality of my house (and heart), to that injur...",A1-A2,uk-key-stage-readability-for-english-texts
2,"Wolves or watch-dogs, it was hard to say from which the sheep had most to fear. The Castle of Villefranche was harsh and stern as its master. A broad moat, ...",C1-C2,uk-key-stage-readability-for-english-texts


# 3. asap-2-0

ASAP 2.0 is a student-essay scoring dataset with prompt-specific human scores and the original essay text.

- Structure: one CSV file with one row per essay; the main text field is `full_text` and the native score field is `score`.
- Observations: 24,728 essays are available locally.
- Labels: prompt-specific human scores (`score`) plus prompt metadata (`prompt_name`).
- Format: full student essays, not isolated words.


In [3]:
asap2_dir = DATASET_DIRS["asap-2-0"]
asap2_raw = pd.read_csv(asap2_dir / "ASAP2_train_sourcetexts.csv")
asap2_raw["score"] = pd.to_numeric(asap2_raw["score"], errors="coerce")

text_col = "full_text"
original_label_col = "score"
group_col = "prompt_name"

show_basic_overview(asap2_raw, text_col=text_col, label_cols=[original_label_col, group_col])
print("Prompt-specific score ranges:")
display(
    asap2_raw.groupby(group_col)[original_label_col]
    .agg(["min", "max", "mean", "count"])
    .sort_index()
    .round(2)
)

print("Normalized score band mapping:", SCORE_CLASS_BANDS)
# Assumption: ASAP 2.0 does not ship with CEFR/readability labels, so
# prompt-normalized scores are used as a low/mid/high proxy for the unified classes.
asap2_prepared = asap2_raw.copy()
asap2_prepared["score_normalized"] = asap2_prepared.groupby(group_col)[original_label_col].transform(min_max_normalize)
asap2_prepared["label"] = asap2_prepared["score_normalized"].apply(normalized_score_to_label)
assert asap2_prepared["label"].notna().all(), "Unmapped ASAP 2.0 score bands found."

display(asap2_prepared[[group_col, original_label_col, "score_normalized", "label"]].head())

asap2_prepared = build_standardized_dataframe(
    asap2_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="asap-2-0",
)
asap2_standardized, asap2_output_path = save_standardized_dataset(
    asap2_prepared,
    "asap-2-0.parquet",
)


Shape: (24728, 14)
Columns: ['essay_id', 'score', 'full_text', 'assignment', 'prompt_name', 'economically_disadvantaged', 'student_disability_status', 'ell_status', 'race_ethnicity', 'gender', 'source_text_1', 'source_text_2', 'source_text_3', 'source_text_4']
Text column: full_text
Label columns: ['score', 'prompt_name']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count    24728.00
mean       362.90
std        148.46
min        150.00
25%        249.00
50%        338.00
75%        446.00
max       1656.00
Name: full_text, dtype: float64

Value counts for score:


score
3    9021
2    6847
4    5553
1    1751
5    1356
6     200
Name: count, dtype: int64

Value counts for prompt_name:


...

Normalized score band mapping: {'A1-A2': (0.0, 0.3333333333333333), 'B1-B2': (0.3333333333333333, 0.6666666666666666), 'C1-C2': (0.6666666666666666, 1.0)}


,prompt_name,score,score_normalized,label
0,Exploring Venus,4,0.6,B1-B2
1,Exploring Venus,2,0.2,A1-A2
2,Exploring Venus,3,0.4,B1-B2
3,Exploring Venus,2,0.2,A1-A2
4,Exploring Venus,2,0.2,A1-A2


Saved data/prepared/asap-2-0.parquet with shape (24728, 3)
Unified label distribution:


label
A1-A2     8598
B1-B2    14368
C1-C2     1762
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"The author suggests that studying Venus is worthy enough even though it is very dangerous. The author mentioned that on the planet's surface, temperatures a...",B1-B2,asap-2-0
1,NASA is fighting to be alble to to go to Venus . They have been researching diffrent methods on how to sustaine life on the planet . In the text it says tha...,A1-A2,asap-2-0
2,"""The Evening Star"", is one of the brightest points of the light on the sky at night. Venus is a planet, Also Venus is the second planet from the sun. Venus ...",B1-B2,asap-2-0


# 4. asap-aes

ASAP-AES is the original Automated Student Assessment Prize essay scoring dataset.

- Structure: a labeled training TSV and unlabeled validation/test TSV files; one row per essay with the essay text in `essay`.
- Observations: 12,976 labeled training essays are available locally, plus unlabeled validation/test splits.
- Labels: `domain1_score` is the native gold score for the training split, grouped by `essay_set` because score ranges vary by prompt.
- Format: full student essays, not isolated words.


In [4]:
asap_aes_dir = DATASET_DIRS["asap-aes"]
asap_aes_raw = pd.read_csv(asap_aes_dir / "training_set_rel3.tsv", sep="	", encoding="latin1")
asap_aes_valid = pd.read_csv(asap_aes_dir / "valid_set.tsv", sep="	", encoding="latin1")
asap_aes_test = pd.read_csv(asap_aes_dir / "test_set.tsv", sep="	", encoding="latin1")

asap_aes_raw["domain1_score"] = pd.to_numeric(asap_aes_raw["domain1_score"], errors="coerce")
asap_aes_raw["essay_set"] = pd.to_numeric(asap_aes_raw["essay_set"], errors="coerce")

text_col = "essay"
original_label_col = "domain1_score"
group_col = "essay_set"

show_basic_overview(asap_aes_raw, text_col=text_col, label_cols=[original_label_col, group_col])
print("Unlabeled split shapes available locally:")
print({"valid": asap_aes_valid.shape, "test": asap_aes_test.shape})
print("Essay-set-specific score ranges:")
display(
    asap_aes_raw.groupby(group_col)[original_label_col]
    .agg(["min", "max", "mean", "count"])
    .sort_index()
    .round(2)
)

print("Normalized score band mapping:", SCORE_CLASS_BANDS)
# Assumption: ASAP-AES does not provide CEFR/readability labels, so
# essay-set-normalized scores are used as a low/mid/high proxy for the unified classes.
asap_aes_prepared = asap_aes_raw.copy()
asap_aes_prepared["score_normalized"] = asap_aes_prepared.groupby(group_col)[original_label_col].transform(min_max_normalize)
asap_aes_prepared["label"] = asap_aes_prepared["score_normalized"].apply(normalized_score_to_label)
assert asap_aes_prepared["label"].notna().all(), "Unmapped ASAP-AES score bands found."

display(asap_aes_prepared[[group_col, original_label_col, "score_normalized", "label"]].head())

asap_aes_prepared = build_standardized_dataframe(
    asap_aes_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="asap-aes",
)
asap_aes_standardized, asap_aes_output_path = save_standardized_dataset(
    asap_aes_prepared,
    "asap-aes.parquet",
)


Shape: (12976, 28)
Columns: ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Text column: essay
Label columns: ['domain1_score', 'essay_set']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count    12976.00
mean       222.71
std        175.92
min          2.00
25%         98.00
50%        163.00
75%        307.00
max       1064.00
Name: essay, dtype: float64

Value counts for domain1_score:


domain1_score
3     2830
2     2445
1     1736
4     1424
8      737
0      418
9      383
10     372
16     199
11     165
7      163
40     161
17     160
6      137
12     133
18     118
14     105
20     103
24      99
5       96
19      88
15      86
13      82
21      70
36      65
22      63
23      53
30      49
35      47
34      39
37      39
32      37
31      34
33      32
45      31
42      23
41      22
38      20
43      15
44      14
46      13
50      13
28      11
39       8
29       8
47       7
27       6
25       5
26       4
48       3
55       2
49       2
60       1
Name: count, dtype: int64

Value counts for essay_set:


...

Normalized score band mapping: {'A1-A2': (0.0, 0.3333333333333333), 'B1-B2': (0.3333333333333333, 0.6666666666666666), 'C1-C2': (0.6666666666666666, 1.0)}


,essay_set,domain1_score,score_normalized,label
0,1,8,0.6,B1-B2
1,1,9,0.7,C1-C2
2,1,7,0.5,B1-B2
3,1,10,0.8,C1-C2
4,1,8,0.6,B1-B2


Saved data/prepared/asap-aes.parquet with shape (12976, 3)
Unified label distribution:


label
A1-A2    2530
B1-B2    6038
C1-C2    4408
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"Dear local newspaper, I think effects computers have on people are great learning skills/affects because they give us time to chat with friends/new people, ...",B1-B2,asap-aes
1,"Dear @CAPS1 @CAPS2, I believe that using computers will benefit us in many ways like talking and becoming friends will others through websites like facebook...",C1-C2,asap-aes
2,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology ...",B1-B2,asap-aes


# 5. cerf-levelled-english-texts

This local folder contains a CEFR-labeled text corpus stored as a simple two-column CSV.

- Structure: one CSV file (`cefr_leveled_texts.csv`) with `text` and `label` columns.
- Observations: 1,494 texts are available locally.
- Labels: CEFR levels `A1`, `A2`, `B1`, `B2`, `C1`, and `C2`.
- Format: full texts/excerpts, not isolated words.


In [5]:
cerf_dir = DATASET_DIRS["cerf-levelled-english-texts"]
cerf_raw = pd.read_csv(cerf_dir / "cefr_leveled_texts.csv")

text_col = "text"
original_label_col = "label"

show_basic_overview(cerf_raw, text_col=text_col, label_cols=[original_label_col])

cerf_label_mapping = {
    "A1": "A1-A2",
    "A2": "A1-A2",
    "B1": "B1-B2",
    "B2": "B1-B2",
    "C1": "C1-C2",
    "C2": "C1-C2",
}
print("Unified mapping:", cerf_label_mapping)

cerf_prepared = cerf_raw.copy()
cerf_prepared["label"] = cerf_prepared[original_label_col].map(cerf_label_mapping)
assert cerf_prepared["label"].notna().all(), "Unmapped CEFR labels found."

cerf_prepared = build_standardized_dataframe(
    cerf_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="cerf-levelled-english-texts",
)
cerf_standardized, cerf_output_path = save_standardized_dataset(
    cerf_prepared,
    "cerf-levelled-english-texts.parquet",
)


Shape: (1494, 2)
Columns: ['text', 'label']
Text column: text
Label columns: ['label']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count    1494.00
mean      414.85
std       324.75
min        34.00
25%       125.00
50%       308.00
75%       617.00
max      2227.00
Name: text, dtype: float64

Value counts for label:


label
A1    288
B2    286
A2    272
C1    241
B1    205
C2    202
Name: count, dtype: int64

Preview:


,text,label
0,"Hi! I've been meaning to write for ages and finally today I'm actually doing something about it. Not that I'm trying to make excuses for myself, it's been r...",B2
1,﻿It was not so much how hard people found the challenge but how far they would go to avoid it that left researchers gobsmacked. The task? To sit in a chair ...,B2
2,"Keith recently came back from a trip to Chicago, Illinois. This midwestern metropolis is found along the shore of Lake Michigan. During his visit, Keith spe...",B2


Unified mapping: {'A1': 'A1-A2', 'A2': 'A1-A2', 'B1': 'B1-B2', 'B2': 'B1-B2', 'C1': 'C1-C2', 'C2': 'C1-C2'}
Saved data/prepared/cerf-levelled-english-texts.parquet with shape (1494, 3)
Unified label distribution:


label
A1-A2    560
B1-B2    491
C1-C2    443
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"Hi! I've been meaning to write for ages and finally today I'm actually doing something about it. Not that I'm trying to make excuses for myself, it's been r...",B1-B2,cerf-levelled-english-texts
1,﻿It was not so much how hard people found the challenge but how far they would go to avoid it that left researchers gobsmacked. The task? To sit in a chair ...,B1-B2,cerf-levelled-english-texts
2,"Keith recently came back from a trip to Chicago, Illinois. This midwestern metropolis is found along the shore of Lake Michigan. During his visit, Keith spe...",B1-B2,cerf-levelled-english-texts


# 6. one-stop-corpus-english

OneStopEnglish is a level-controlled news corpus with parallel article versions for different learner levels.

- Structure: plain-text files in three folders (`Ele`, `Int`, `Adv`), one text file per article version.
- Observations: 509 texts are available locally (178 elementary + 170 intermediate + 161 advanced).
- Labels: folder-level difficulty markers `Ele`, `Int`, and `Adv`.
- Format: full rewritten articles, not isolated words.


In [6]:
one_stop_dir = DATASET_DIRS["one-stop-corpus-english"]
one_stop_raw = load_folder_corpus(one_stop_dir, ["Ele", "Int", "Adv"], label_column_name="level")

text_col = "text"
original_label_col = "level"

show_basic_overview(one_stop_raw, text_col=text_col, label_cols=[original_label_col])
print("File counts by level:")
display(one_stop_raw[original_label_col].value_counts().rename("count"))

one_stop_label_mapping = {
    "Ele": "A1-A2",
    "Int": "B1-B2",
    "Adv": "C1-C2",
}
print("Unified mapping:", one_stop_label_mapping)

one_stop_prepared = one_stop_raw.copy()
one_stop_prepared["label"] = one_stop_prepared[original_label_col].map(one_stop_label_mapping)
assert one_stop_prepared["label"].notna().all(), "Unmapped OneStop levels found."

one_stop_prepared = build_standardized_dataframe(
    one_stop_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="one-stop-corpus-english",
)
one_stop_standardized, one_stop_output_path = save_standardized_dataset(
    one_stop_prepared,
    "one-stop-corpus-english.parquet",
)


Shape: (509, 3)
Columns: ['document_id', 'level', 'text']
Text column: text
Label columns: ['level']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count     509.00
mean      683.25
std       178.76
min       259.00
25%       566.00
50%       671.00
75%       786.00
max      1368.00
Name: text, dtype: float64

Value counts for level:


level
Ele    178
Int    170
Adv    161
Name: count, dtype: int64

Preview:


...

File counts by level:


level
Ele    178
Int    170
Adv    161
Name: count, dtype: int64

Unified mapping: {'Ele': 'A1-A2', 'Int': 'B1-B2', 'Adv': 'C1-C2'}
Saved data/prepared/one-stop-corpus-english.parquet with shape (509, 3)
Unified label distribution:


label
A1-A2    178
B1-B2    170
C1-C2    161
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,Scientists have connected the brains of two rats and allowed them to share information. Researchers say this is an important step towards creating the world...,A1-A2,one-stop-corpus-english
1,Scientists have put a false memory in the brains of mice in an experiment. They hopethe results of the experiment will help to explain why people “remember”...,A1-A2,one-stop-corpus-english
2,"The senior editor of The Atlantic magazine, James Hamblin, recently did an experiment. As part of his series, ‘If Our Bodies Could Talk’, Hamblin reduced th...",A1-A2,one-stop-corpus-english


# 7. cambridge-english-readability

This corpus is organized by Cambridge exam level, which acts as the native difficulty label.

- Structure: plain-text files in five level folders (`KET`, `PET`, `FCE`, `CAE`, `CPE`), one text file per reading passage.
- Observations: 282 texts are available locally (61 KET + 55 PET + 67 FCE + 55 CAE + 44 CPE).
- Labels: Cambridge exam levels `KET`, `PET`, `FCE`, `CAE`, and `CPE`.
- Format: full texts/exam passages, not isolated words.


In [7]:
cambridge_dir = DATASET_DIRS["cambridge-english-readability"]
cambridge_raw = load_folder_corpus(
    cambridge_dir,
    ["KET", "PET", "FCE", "CAE", "CPE"],
    label_column_name="exam_level",
)

text_col = "text"
original_label_col = "exam_level"

show_basic_overview(cambridge_raw, text_col=text_col, label_cols=[original_label_col])
print("File counts by exam level:")
display(cambridge_raw[original_label_col].value_counts().rename("count"))

cambridge_label_mapping = {
    "KET": "A1-A2",  # KET is roughly A2 on the Cambridge scale.
    "PET": "B1-B2",  # PET is roughly B1.
    "FCE": "B1-B2",  # FCE is roughly B2.
    "CAE": "C1-C2",  # CAE is roughly C1.
    "CPE": "C1-C2",  # CPE is roughly C2.
}
print("Unified mapping:", cambridge_label_mapping)

cambridge_prepared = cambridge_raw.copy()
cambridge_prepared["label"] = cambridge_prepared[original_label_col].map(cambridge_label_mapping)
assert cambridge_prepared["label"].notna().all(), "Unmapped Cambridge levels found."

cambridge_prepared = build_standardized_dataframe(
    cambridge_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="cambridge-english-readability",
)
cambridge_standardized, cambridge_output_path = save_standardized_dataset(
    cambridge_prepared,
    "cambridge-english-readability.parquet",
)


Shape: (282, 3)
Columns: ['document_id', 'exam_level', 'text']
Text column: text
Label columns: ['exam_level']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count     282.00
mean      511.59
std       306.74
min        76.00
25%       197.25
50%       563.00
75%       734.50
max      1322.00
Name: text, dtype: float64

Value counts for exam_level:


exam_level
FCE    67
KET    61
PET    55
CAE    55
CPE    44
Name: count, dtype: int64

Preview:


...

File counts by exam level:


exam_level
FCE    67
KET    61
PET    55
CAE    55
CPE    44
Name: count, dtype: int64

Unified mapping: {'KET': 'A1-A2', 'PET': 'B1-B2', 'FCE': 'B1-B2', 'CAE': 'C1-C2', 'CPE': 'C1-C2'}
Saved data/prepared/cambridge-english-readability.parquet with shape (282, 3)
Unified label distribution:


label
A1-A2     61
B1-B2    122
C1-C2     99
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"THE HISTORY OF THE LONDON POLICE Today there are policemen everywhere, but in 1700 London had no policemen at all. A few old men used to protect the city st...",A1-A2,cambridge-english-readability
1,"Dear Deshini, It's great that you are my new penfriend. My name is Tom and I am fifteen years old. I was born in Canada but I live in England now. Please wr...",A1-A2,cambridge-english-readability
2,"REBECCCA STEVENS Rebecca Stevens was the first woman to climb Mount Everest. Before she went up the highest mountain in the world, she was a journalist and ...",A1-A2,cambridge-english-readability


# 8. commonlit-readability-prize

The local CommonLit resource is the CLEAR corpus workbook, a readability-oriented collection of literary and informational excerpts with metadata and difficulty indicators.

- Structure: one Excel workbook (`CLEAR_corpus_final.xlsx`) with a main `Data` sheet; the text field is `Excerpt`.
- Observations: 4,724 excerpts are available locally.
- Labels: the nearest native difficulty label is `Lexile Band`, which is continuous/banded rather than CEFR.
- Format: full text excerpts, not isolated words.


In [3]:
load_clear_corpus

PosixPath('data/сommonlit-readability-prize')

PosixPath('data/сommonlit-readability-prize')

In [5]:
commonlit_dir = DATASET_DIRS["commonlit-readability-prize"]
commonlit_raw = load_clear_corpus(commonlit_dir / "CLEAR_corpus_final.xlsx", sheet_name="Data")
commonlit_raw.columns = [str(column).strip() for column in commonlit_raw.columns]

openpyxl is not available; using the XML fallback loader for CLEAR_corpus_final.xlsx.


In [7]:
commonlit_raw.head(1)

,ID,Author,Title,Anthology,URL,Pub Year,Categ,Sub Cat,Lexile Band,Location,License,MPAA Max,MPAA #Max,MPAA# Avg,Excerpt,Google WC,Sentence Count,Paragraphs,BT_easiness,s.e.,Flesch-Reading-Ease,Flesch-Kincaid-Grade-Level,Automated Readability Index,SMOG Readability,New Dale-Chall Readability Formula,CAREC,CAREC_M,CML2RI
0,400,Carolyn Wells,Patty's Suitors,NaN,http://www.gutenberg.org/cache/epub/5631/pg5631-images.html,1914,Lit,NaN,900,mid,NaN,G,1,1,"When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.\nThe f...",174,11,6,-0.340259125,0.46400904599999998,81.7,5.95,7.37,8,6.55,0.12102,0.11952,12.097815499999999


In [8]:
text_col = "Excerpt"
original_label_col = "Lexile Band"

numeric_columns = [
    "Pub Year",
    "Google WC",
    "Sentence Count",
    "Paragraphs",
    "BT_easiness",
    "s.e.",
    "Flesch-Reading-Ease",
    "Flesch-Kincaid-Grade-Level",
    "Automated Readability Index",
    "SMOG Readability",
    "New Dale-Chall Readability Formula",
    "CAREC",
    "CAREC_M",
    "CML2RI",
]
for column in numeric_columns:
    if column in commonlit_raw.columns:
        commonlit_raw[column] = pd.to_numeric(commonlit_raw[column], errors="coerce")

In [9]:
show_basic_overview(commonlit_raw, text_col=text_col, label_cols=[original_label_col, "Categ", "Location"])

Shape: (4724, 28)
Columns: ['ID', 'Author', 'Title', 'Anthology', 'URL', 'Pub Year', 'Categ', 'Sub Cat', 'Lexile Band', 'Location', 'License', 'MPAA Max', 'MPAA #Max', 'MPAA# Avg', 'Excerpt', 'Google WC', 'Sentence Count', 'Paragraphs', 'BT_easiness', 's.e.', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CML2RI']
Text column: Excerpt
Label columns: ['Lexile Band', 'Categ', 'Location']
Observation format: full_texts_or_excerpts
Text length statistics (word counts):


count    4724.00
mean      173.34
std        17.00
min       129.00
25%       160.00
50%       175.00
75%       188.00
max       205.00
Name: Excerpt, dtype: float64

Value counts for Lexile Band:


Lexile Band
1100           1379
1300           1359
900             815
700             478
500             364
1500            186
1700             58
300              41
410L-600L        12
610L-800L        12
1900              9
410L - 600L       6
100               3
610L - 800L       1
610L-800          1
Name: count, dtype: int64

Value counts for Categ:


Categ
Lit     2420
Info    2304
Name: count, dtype: int64

Value counts for Location:


Location
mid      3470
start    1024
whole     122
end       108
Name: count, dtype: int64

Preview:


,Excerpt,Lexile Band,Categ,Location
0,"When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape. The fl...",900,Lit,mid
1,"All through dinner time, Mrs. Fayre was somewhat silent, her eyes resting on Dolly with a wistful, uncertain expression. She wanted to give the child the pl...",700,Lit,mid
2,"As Roger had predicted, the snow departed as quickly as it came, and two days after their sleigh ride there was scarcely a vestige of white on the ground. T...",900,Lit,mid


In [10]:
print("Lexile band counts:")
display(commonlit_raw[original_label_col].value_counts(dropna=False).rename("count"))

Lexile band counts:


Lexile Band
1100           1379
1300           1359
900             815
700             478
500             364
1500            186
1700             58
300              41
410L-600L        12
610L-800L        12
1900              9
410L - 600L       6
100               3
610L - 800L       1
610L-800          1
Name: count, dtype: int64

In [11]:
print("Selected readability statistics:")
display(
    commonlit_raw[["Google WC", "Sentence Count", "Paragraphs", "Flesch-Kincaid-Grade-Level"]]
    .describe()
    .round(2)
)

Selected readability statistics:


,Google WC,Sentence Count,Paragraphs,Flesch-Kincaid-Grade-Level
count,4724.00,4724.00,4724.00,4724.00
mean,171.96,9.57,2.54,9.51
std,16.99,4.64,1.87,4.33
min,125.00,2.00,1.00,-1.04
25%,158.00,7.00,1.00,6.56
50%,174.00,8.00,2.00,9.35
75%,186.00,11.00,3.00,11.97
max,205.00,41.00,20.00,42.64


In [12]:
print("Lexile mapping thresholds:", LEXILE_CLASS_BANDS)
# Assumption: CLEAR is not CEFR-labeled. Lexile band is the closest native difficulty signal,
# so coarse lexile thresholds are used as an editable proxy for the unified classes.

Lexile mapping thresholds: {'A1-A2': {'max': 700}, 'B1-B2': {'min': 700, 'max': 1100}, 'C1-C2': {'min': 1100}}


In [38]:
commonlit_prepared = commonlit_raw.copy()
commonlit_prepared["lexile_numeric"] = commonlit_prepared[original_label_col].apply(extract_lexile_value)
commonlit_prepared["label"] = commonlit_prepared["lexile_numeric"].apply(lexile_to_label)
assert commonlit_prepared["label"].notna().all(), "Unmapped CLEAR lexile values found."

display(commonlit_prepared[[original_label_col, "lexile_numeric", "label"]].head())

,Lexile Band,lexile_numeric,label
0,900,900.0,B1-B2
1,700,700.0,A1-A2
2,900,900.0,B1-B2
3,1300,1300.0,C1-C2
4,1300,1300.0,C1-C2


In [40]:
commonlit_prepared = build_standardized_dataframe(
    commonlit_prepared,
    text_col=text_col,
    label_col="label",
    source_dataset="commonlit-readability-prize",
)

In [41]:
commonlit_prepared.head()

,text,label,source_dataset
0,"When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.\nThe f...",B1-B2,commonlit-readability-prize
1,"All through dinner time, Mrs. Fayre was somewhat silent, her eyes resting on Dolly with a wistful, uncertain expression. She wanted to give the child the pl...",A1-A2,commonlit-readability-prize
2,"As Roger had predicted, the snow departed as quickly as it came, and two days after their sleigh ride there was scarcely a vestige of white on the ground. T...",B1-B2,commonlit-readability-prize
3,"Mr. Grimes was to come up next morning to Sir John Harthover's, at the Place, for his old chimney-sweep was gone to prison, and the chimneys wanted sweeping...",C1-C2,commonlit-readability-prize
4,"And outside before the palace a great garden was walled round, filled full of stately fruit-trees, gray olives and sweet figs, and pomegranates, pears, and ...",C1-C2,commonlit-readability-prize


In [43]:
commonlit_standardized, commonlit_output_path = save_standardized_dataset(
    commonlit_prepared,
    "commonlit-readability-prize.parquet",
)

Saved data/prepared/commonlit-readability-prize.parquet with shape (4724, 3)
Unified label distribution:


label
A1-A2     904
B1-B2    2208
C1-C2    1612
Name: count, dtype: int64[pyarrow]

,text,label,source_dataset
0,"When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape. The fl...",B1-B2,commonlit-readability-prize
1,"All through dinner time, Mrs. Fayre was somewhat silent, her eyes resting on Dolly with a wistful, uncertain expression. She wanted to give the child the pl...",A1-A2,commonlit-readability-prize
2,"As Roger had predicted, the snow departed as quickly as it came, and two days after their sleigh ride there was scarcely a vestige of white on the ground. T...",B1-B2,commonlit-readability-prize


# 9. Result

This section loads every standardized parquet file, prints dataset shapes and label distributions, concatenates them into a single dataframe, and saves the combined result.


In [9]:
prepared_files = {
    "uk-key-stage-readability-for-english-texts": uk_output_path,
    "asap-2-0": asap2_output_path,
    "asap-aes": asap_aes_output_path,
    "cerf-levelled-english-texts": cerf_output_path,
    "one-stop-corpus-english": one_stop_output_path,
    "cambridge-english-readability": cambridge_output_path,
    "commonlit-readability-prize": commonlit_output_path,
}

prepared_frames = []
for dataset_name, parquet_path in prepared_files.items():
    dataset_df = pd.read_parquet(parquet_path, engine=PARQUET_ENGINE)
    print(f"\n{dataset_name}: {dataset_df.shape}")
    print("Label distribution:")
    display(dataset_df["label"].value_counts().reindex(UNIFIED_LABELS, fill_value=0))
    prepared_frames.append(dataset_df)

combined_df = pd.concat(prepared_frames, ignore_index=True)
combined_output_path = OUTPUT_DIR / "all-datasets-unified.parquet"
combined_df.to_parquet(combined_output_path, index=False, engine=PARQUET_ENGINE)

print("\nCombined dataframe shape:", combined_df.shape)
print("Combined label distribution:")
display(combined_df["label"].value_counts().reindex(UNIFIED_LABELS, fill_value=0))
print("Combined source dataset distribution:")
display(combined_df["source_dataset"].value_counts())
display(combined_df.sample(min(5, len(combined_df)), random_state=42))
print(f"Saved combined parquet to: {combined_output_path}")



uk-key-stage-readability-for-english-texts: (20000, 3)
Label distribution:


label
A1-A2     5000
B1-B2    10000
C1-C2     5000
Name: count, dtype: int64[pyarrow]


asap-2-0: (24728, 3)
Label distribution:


label
A1-A2     8598
B1-B2    14368
C1-C2     1762
Name: count, dtype: int64[pyarrow]


asap-aes: (12976, 3)
Label distribution:


...

label
A1-A2    17831
B1-B2    33397
C1-C2    13485
Name: count, dtype: int64[pyarrow]

Combined source dataset distribution:


source_dataset
asap-2-0                                      24728
uk-key-stage-readability-for-english-texts    20000
asap-aes                                      12976
commonlit-readability-prize                    4724
cerf-levelled-english-texts                    1494
one-stop-corpus-english                         509
cambridge-english-readability                   282
Name: count, dtype: int64

,text,label,source_dataset
31546,"""Paul"", Luke said ""do you want to join the Seagoing Cowboys program?"" ""Uhm I'm not sure luke, why would I join the Seagoing Cowboys program, like give me an...",A1-A2,asap-2-0
28786,"I think that the ""Facial Action Coding System"" may actually be valuable. The machine detects emotions from using a picture and that can help with knowing ho...",B1-B2,asap-2-0
54208,"The builders of the Empire State Building were challenged the obstacles of construction, safety, and nature that did not allow dirigibles to dock there. The...",C1-C2,asap-aes
13069,"Then, as their beds were quite close to each other, to stand between them in the form of a green, icy-cold corpse, till they became paralysed with fear, and...",A1-A2,uk-key-stage-readability-for-english-texts
28877,The Facial Action Coding System would be a vaulubale asset in a classroom but it could be intrusive to others peoples feelings and emotions. There are few g...,B1-B2,asap-2-0


Saved combined parquet to: data/prepared/all-datasets-unified.parquet
